# S1 · 02 · Herramientas y acciones
Tres ejemplos mínimos, los mismos de las **Demos 2, 3 y 4** (una tool, una tool con datos y una escritura con permiso); luego tus **Ejercicios 2 a 5** (≈ 45 min; el 5 completa tu `mi_copiloto/`).
Corre sin red ni LLM (`COURSE_OFFLINE=1`, por defecto). Datos ficticios: «hoy» es el martes 14 de mayo de 2030.


## Preparación


In [ ]:
import json
from dataclasses import dataclass
from pathlib import Path

from pydantic_ai import Agent, DeferredToolRequests, DeferredToolResults, RunContext
from pydantic_ai.messages import ToolCallPart, ToolReturnPart
from utilidades.config import crear_modelo, describir_ruta
from utilidades.practica import preparar, revisar, ver_mensajes

settings, OFFLINE = preparar()
modelo = crear_modelo(settings, OFFLINE)
print("Modelo activo:", describir_ruta(settings, OFFLINE))

DATOS = Path("mi_copiloto/datos")  # agenda y correo ficticios de Alex


def partes(r, tipo) -> list:
    """Las partes de un tipo (ToolCallPart, ToolReturnPart…) en todos los mensajes de un run."""
    return [p for m in r.all_messages() for p in m.parts if isinstance(p, tipo)]


## Ejemplo mínimo 1 · Una tool sin contexto: `tool_plain` (Demo 2)
Una **tool** es una función de Python que el modelo puede **pedir** que se ejecute. El modelo no ve tu código: ve el **nombre**, el **docstring** (es el prompt de la tool) y el esquema de los argumentos, que Pydantic AI saca de los tipos de la firma (`monto_bs: float`).
- `@agente.tool_plain` registra una función que **no necesita nada del programa**: solo sus argumentos.
- El recorrido es: el modelo pide la tool (🤖) → Pydantic AI la ejecuta y devuelve el resultado (🔧) → el modelo responde con ese dato.


In [ ]:
agente_cambio = Agent(modelo, instructions="Responde en español, breve. Usa las herramientas para calcular.")


@agente_cambio.tool_plain
def bs_a_usd(monto_bs: float) -> float:
    """Convierte un monto en bolivianos (Bs) a dólares (USD) al tipo de cambio oficial 6.96."""
    return round(monto_bs / 6.96, 2)


r = await agente_cambio.run("¿Cuántos dólares son 700 bolivianos?")
ver_mensajes(r)
print(r.output)


## Ejemplo mínimo 2 · Una tool con datos: `deps`, `RunContext` y `@agente.tool` (Demo 3)
Muchas tools necesitan algo **del programa**, no del modelo: una agenda, una conexión, el usuario. Eso son las **dependencias**:
- `Deps` es una `dataclass` con lo que el agente necesita (aquí, la lista de eventos). `deps_type=Deps` le dice al agente qué tipo esperar.
- En cada `run(..., deps=Deps(...))` el programa entrega sus dependencias.
- `@agente.tool` (sin `_plain`) recibe como primer argumento `ctx: RunContext[Deps]`, y `ctx.deps` es ese objeto. El modelo **no ve** `ctx`: solo `fecha`.
- La fecha de hoy va en las instrucciones, para que el modelo convierta «mañana» en una fecha.


In [ ]:
@dataclass
class Deps:
    agenda: list[dict]


AGENDA = json.loads((DATOS / "agenda.json").read_text(encoding="utf-8"))

agente_agenda = Agent(
    modelo,
    deps_type=Deps,
    instructions="Eres el asistente de Alex. Responde en español, breve. Hoy es martes 2030-05-14.",
)


@agente_agenda.tool
def consultar_agenda(ctx: RunContext[Deps], fecha: str) -> list[dict]:
    """Devuelve los eventos de la agenda de Alex en un día (fecha en formato AAAA-MM-DD)."""
    return [e for e in ctx.deps.agenda if e["fecha"] == fecha]


r = await agente_agenda.run("¿Qué tengo en la agenda el 2030-05-16?", deps=Deps(agenda=AGENDA))
ver_mensajes(r)
print(r.output)


## Ejemplo mínimo 3 · Escribir con permiso: `requires_approval` (Demo 4)
Leer es libre; **escribir** pide permiso. Con `@agente.tool(requires_approval=True)` el modelo puede **pedir** la tool, pero el run **termina sin ejecutarla** y devuelve un `DeferredToolRequests` (la lista de llamadas que esperan decisión). Por eso `output_type=[str, DeferredToolRequests]`: el run acaba con texto **o** con acciones pendientes.
La decisión la toma una persona, **fuera del modelo**: `DeferredToolResults(approvals={id: True})` aprueba (`False` rechaza) y un segundo `run` con `message_history=r.all_messages()` continúa donde quedó, con las mismas `deps`.


In [ ]:
agente_escritor = Agent(
    modelo,
    deps_type=Deps,
    output_type=[str, DeferredToolRequests],
    instructions="Eres el asistente de Alex. Responde en español, breve. Hoy es martes 2030-05-14.",
)


@agente_escritor.tool(requires_approval=True)
def crear_evento(ctx: RunContext[Deps], titulo: str, fecha: str, inicio: str, fin: str) -> dict:
    """Crea un evento en la agenda de Alex (fecha AAAA-MM-DD, horas HH:MM). Requiere aprobación."""
    evento = {"fecha": fecha, "inicio": inicio, "fin": fin, "titulo": titulo}
    ctx.deps.agenda.append(evento)
    return evento


deps = Deps(agenda=list(AGENDA))  # una copia: no tocamos AGENDA
r = await agente_escritor.run("Crea «Llamada con Ana» el 2030-05-16 de 15:00 a 16:00", deps=deps)
print("El run terminó con:", type(r.output).__name__)
if isinstance(r.output, DeferredToolRequests):
    for llamada in r.output.approvals:
        print("Pide:", llamada.tool_name, llamada.args_as_dict())
print("Eventos en la agenda:", len(deps.agenda))


Ahora decidimos: aprobar ejecuta la tool y el modelo continúa con su resultado.


In [ ]:
if isinstance(r.output, DeferredToolRequests):
    decisiones = {llamada.tool_call_id: True for llamada in r.output.approvals}
    r = await agente_escritor.run(
        message_history=r.all_messages(),
        deferred_tool_results=DeferredToolResults(approvals=decisiones),
        deps=deps,
    )
    print(r.output)
else:  # un modelo pequeño a veces responde sin pedir la tool: repite el pedido
    print("El modelo no pidió crear_evento esta vez.")
print("Eventos en la agenda:", len(deps.agenda))


## Ejercicio 2 · Tu primera tool
**Se apoya en:** Demo 2 · **Nivel:** básico · ≈ 8 min. **Qué cambia:** en la demo la tool convertía moneda; aquí escribes otra que calcula con horas.

1. Escribe `minutos_entre(inicio, fin)`: recibe dos horas `"HH:MM"` y devuelve los minutos entre ellas (`int`). Pista: `"09:15".split(":")` da `["09", "15"]`.
2. Ponle un **docstring** que diga qué hace y el formato de las horas.
3. Regístrala con `@agente_minutos.tool_plain`.


In [ ]:
agente_minutos = Agent(modelo, instructions="Responde en español, breve. Usa las herramientas para calcular.")

# TODO Ejercicio 2: decora con @agente_minutos.tool_plain, escribe el docstring y el cuerpo.
def minutos_entre(inicio: str, fin: str) -> int:
    return 0


r2 = await agente_minutos.run("¿Cuántos minutos hay entre 09:15 y 11:45?")
ver_mensajes(r2)
print(r2.output)


In [ ]:
revisar("Ejercicio 2 · la función calcula bien", lambda: minutos_entre("09:15", "11:45") == 150, "De 09:15 a 11:45 hay 150 minutos")
revisar("Ejercicio 2 · tiene docstring", lambda: len(minutos_entre.__doc__ or "") > 20, "El docstring es el prompt de la tool: explica qué hace y el formato")
revisar(
    "Ejercicio 2 · el modelo llamó la tool",
    lambda: any(p.tool_name == "minutos_entre" for p in partes(r2, ToolCallPart)),
    "Registra la función con @agente_minutos.tool_plain y vuelve a ejecutar",
)
revisar(
    "Ejercicio 2 · la tool devolvió 150",
    lambda: any(p.tool_name == "minutos_entre" and p.content == 150 for p in partes(r2, ToolReturnPart)),
    "Con IA real, si el modelo pasó otras horas, repite la pregunta",
)


## Ejercicio 3 · Una tool que lee datos
**Se apoya en:** Demo 3 · **Nivel:** medio · ≈ 12 min. **Qué cambia:** en la demo la tool leía la agenda; aquí lees el **correo** y devuelves una `fuente` que el agente pueda citar.

1. Completa `DepsCorreo` con un campo `correos: list[dict]`.
2. Completa `buscar_correos(ctx, palabra)`: recorre `ctx.deps.correos` y, por cada correo cuyo `asunto` o `cuerpo` contenga `palabra` (sin distinguir mayúsculas), devuelve solo `de`, `asunto` y `fuente=f"correo:{id}"`. No devuelvas el cuerpo entero: el modelo no lo necesita.
3. Ejecuta la pregunta con `deps=DepsCorreo(correos=CORREOS)`.

Con el simulado verás que pasa la frase entera como `palabra` y recibe `[]`: no entiende el pedido. Por eso la autocorrección prueba tu función también sola, con palabras concretas. Con IA real deberías ver `palabra="presupuesto"`.


In [ ]:
CORREOS = json.loads((DATOS / "correo.json").read_text(encoding="utf-8"))
print("Un correo de ejemplo:", CORREOS[0])


@dataclass
class DepsCorreo:
    # TODO Ejercicio 3: añade el campo correos (list[dict]).
    pass


agente_correo = Agent(
    modelo,
    deps_type=DepsCorreo,
    instructions="Eres el asistente de Alex. Responde en español, breve. Usa las herramientas para leer el correo.",
)


@agente_correo.tool
def buscar_correos(ctx: RunContext[DepsCorreo], palabra: str) -> list[dict]:
    """Busca correos de Alex que contengan una palabra en el asunto o el cuerpo.

    Args:
        palabra: Una sola palabra clave, por ejemplo «presupuesto».
    """
    # TODO Ejercicio 3: filtra ctx.deps.correos por `palabra` y devuelve de, asunto y fuente=f"correo:{id}".
    return []


try:
    r3 = await agente_correo.run("Busca mis correos sobre el presupuesto", deps=DepsCorreo(correos=CORREOS))
    ver_mensajes(r3)
    print(r3.output)
except TypeError as exc:  # DepsCorreo todavía no tiene el campo correos
    r3 = None
    print("Todavía no se puede ejecutar:", exc)


In [ ]:
def buscar(palabra: str) -> list[dict]:
    """Prueba la función sola, sin modelo: un `ctx` de mentira que solo trae `deps`."""
    ctx = type("Ctx", (), {"deps": DepsCorreo(correos=CORREOS)})()
    return buscar_correos(ctx, palabra)


revisar(
    "Ejercicio 3 · el modelo llamó buscar_correos",
    lambda: any(p.tool_name == "buscar_correos" for p in partes(r3, ToolCallPart)),
    "Completa DepsCorreo y vuelve a ejecutar",
)
revisar(
    "Ejercicio 3 · encuentra el correo del presupuesto",
    lambda: [c["fuente"] for c in buscar("Presupuesto")] == ["correo:m1"],
    "Filtra por palabra en asunto o cuerpo, sin distinguir mayúsculas",
)
revisar(
    "Ejercicio 3 · cada resultado trae de, asunto y fuente, sin el cuerpo",
    lambda: len(buscar("jueves")) > 0 and all(set(c) == {"de", "asunto", "fuente"} for c in buscar("jueves")),
    'Devuelve {"de": ..., "asunto": ..., "fuente": f"correo:{id}"}',
)


## Ejercicio 4 · Sin permiso no hay acción
**Se apoya en:** Demo 4 · **Nivel:** medio · ≈ 12 min. **Qué cambia:** en la demo solo se aprobaba; aquí primero **rechazas** y compruebas que la agenda no cambió, y luego apruebas.

Usamos `agente_escritor` del ejemplo mínimo 3. El pedido usa el verbo «Crea» para que el agente pida `crear_evento`.


In [ ]:
deps4 = Deps(agenda=list(AGENDA))
antes = len(deps4.agenda)
PEDIDO4 = "Crea «Café con Luis» el 2030-05-17 de 10:00 a 11:00"
r4 = await agente_escritor.run(PEDIDO4, deps=deps4)
pendiente = isinstance(r4.output, DeferredToolRequests)
print("¿Quedó pendiente de aprobación?", pendiente)


**4a · Rechaza.** Arma `DeferredToolResults(approvals={id: False})` con el `tool_call_id` de cada llamada en `r4.output.approvals`, reanuda el run y guarda en `tras_rechazo` cuántos eventos hay ahora.


In [ ]:
tras_rechazo = None
if pendiente:
    # TODO Ejercicio 4a: decisiones = {llamada.tool_call_id: False for llamada in r4.output.approvals}
    # reanuda con agente_escritor.run(message_history=r4.all_messages(), deferred_tool_results=..., deps=deps4)
    # y luego: tras_rechazo = len(deps4.agenda)
    pass
print("Eventos antes:", antes, "· tras rechazar:", tras_rechazo)


**4b · Aprueba.** Pide lo mismo otra vez (un run nuevo con `PEDIDO4` y `deps4`), aprueba con `True`, reanuda y guarda en `tras_aprobacion` cuántos eventos hay.


In [ ]:
tras_aprobacion = None
# TODO Ejercicio 4b: r = await agente_escritor.run(PEDIDO4, deps=deps4); si r.output es DeferredToolRequests,
# aprueba cada llamada con True, reanuda como en 4a y guarda tras_aprobacion = len(deps4.agenda).
print("Eventos antes:", antes, "· tras aprobar:", tras_aprobacion)


In [ ]:
revisar("Ejercicio 4 · la escritura quedó pendiente", lambda: pendiente, "El run debe terminar con DeferredToolRequests (con IA real, repite si el modelo no pidió la tool)")
revisar("Ejercicio 4 · rechazar no guarda nada", lambda: tras_rechazo == antes, "Reanuda con approvals={id: False} y cuenta len(deps4.agenda)")
revisar("Ejercicio 4 · aprobar guarda un evento", lambda: tras_aprobacion == antes + 1, "Pide otra vez, aprueba con True y vuelve a contar")


## Ejercicio 5 · Completa tu copiloto
**Se apoya en:** Demo 4 · **Nivel:** básico · ≈ 8 min. **Qué cambia:** pasas del notebook a un archivo del proyecto que luego usas desde la terminal.

Abre `mi_copiloto/agente.py` y completa los dos `# TODO Ejercicio 5`:
1. el cuerpo de `crear_evento`: guarda el evento en `ctx.deps.eventos_nuevos` y devuelve un dict con el evento y una `fuente` (como en el ejemplo mínimo 3);
2. en `crear_agente(modelo)`, la lista `tools=[...]`: las tres de lectura tal cual y `Tool(crear_evento, requires_approval=True)` (la misma idea que `@agente.tool(requires_approval=True)`, escrita como lista).

Luego conversa con tu agente en una terminal, desde `s1/` (añade `--simulado` si no tienes modelo):
```bash
uv run python -m mi_copiloto.chat
tú › ¿Qué tengo el 2030-05-14?
tú › Crea «Llamada con Ana» el 2030-05-16 de 15:00 a 16:00     → ¿Apruebas crear_evento(...)? [s/N]
```
La celda siguiente comprueba tu archivo sin romper nada (si lo editas, vuelve a ejecutarla).


In [ ]:
import importlib

import mi_copiloto.agente as archivo

archivo = importlib.reload(archivo)  # recoge tus cambios en agente.py
deps_chequeo = archivo.Deps()


def tools_del_archivo() -> dict:
    return archivo.crear_agente(modelo)._function_toolset.tools


def crear_evento_directo() -> int:
    ctx = type("Ctx", (), {"deps": deps_chequeo})()
    archivo.crear_evento(ctx, "Prueba", "2030-05-16", "15:00", "16:00")
    return len(deps_chequeo.eventos_nuevos)


revisar("Ejercicio 5 · crear_evento guarda el evento", lambda: crear_evento_directo() == 1, "Completa el cuerpo de crear_evento en agente.py")
revisar("Ejercicio 5 · el agente tiene las cuatro tools", lambda: len(tools_del_archivo()) == 4, "Registra consultar_agenda, buscar_correos, clima y crear_evento en crear_agente")
revisar("Ejercicio 5 · escribir exige aprobación", lambda: tools_del_archivo()["crear_evento"].requires_approval, "Envuelve crear_evento con Tool(..., requires_approval=True)")


## Cierre
**¿Qué puede hacer ahora Mi copiloto?** Calcular con tools, consultar agenda y correo con **fuentes** y **pedir permiso antes de escribir**. Una pregunta para llevarte: ¿qué decidiste **no** dejarle hacer sin permiso?
**Opcional, con IA real:** `03_ia_real.ipynb` (reto: la misma pregunta con dos modelos).
